### clinvar missense prediction w/ feature intersection
* only use consistent positions
* only missense clinvar
* use positions w/ mpc **OR** pathogenic fraction

In [ ]:
import pandas, numpy
import pydot, pydotplus, graphviz
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model, metrics, tree, svm
from sklearn.neural_network import MLPClassifier
from sklearn.externals.six import StringIO
from IPython.display import HTML
%matplotlib inline

In [ ]:
def calc_path_frac(rows):
    pfam = list(rows['pfam'].values)[0]
    pathogenic = len(rows[ (rows.clin_class=='PATHOGENIC') | (rows.clin_class=='LIKLEY_PATHOGENIC')])
    benign = len(rows[ (rows.clin_class=='LIKELY_BENIGN') | (rows.clin_class=='BENIGN')])
    frac = -1
    if pathogenic+benign:
        frac = pathogenic/(pathogenic+benign)
    return pandas.Series([frac, len(rows)], index=['path_frac', 'size'])

dat_file = '../data/interim/EPIv6.eff.dbnsfp.anno.hHack.dat.xls'
df_pre = pandas.read_csv(dat_file, sep='\t').fillna(0)
df = (df_pre['pfam'].str.split(',', expand=True)
     .stack()
     .reset_index(level=0)
     .set_index('level_0')
     .rename(columns={0:'pfam'})
     .join(df_pre.drop('pfam',1), how='left')
     )
dd = df.groupby('pfam').apply(calc_path_frac)
ff = dd.reset_index()

# mk domain features
def match(row, domain_info):
    ls = []
    for pfam in row['pfam'].split(','):
        if pfam in domain_info:
            if domain_info[pfam][2] == 0:
                ls.append(domain_info[pfam])
    if len(ls) == 0:
        for pfam in row['pfam'].split(','):
            if pfam in domain_info:
                return domain_info[pfam]
        
    if len(ls):
        return ls[0]
    else:
        return (0, 0, 1)
    
ff.loc[:, 'path_na'] = ff.apply(lambda row: 1 if row['path_frac']==-1 else 0, axis=1)
domain_info = {pfam:[path_frac, size, path_na]
               for pfam, path_frac, size, path_na
               in ff.values}

df_pre.loc[:, 'path_frac_t'] = df_pre.apply(lambda row: match(row, domain_info)[0], axis=1)
df_pre.loc[:, 'size_t'] = df_pre.apply(lambda row: match(row, domain_info)[1], axis=1)
df_pre.loc[:, 'path_na_t'] = df_pre.apply(lambda row: match(row, domain_info)[2], axis=1)
df_pre.loc[:, 'in_none_pfam'] = df_pre.apply(lambda row: 1 if 'none' in df_pre['pfam'] else 0, axis=1)

In [ ]:
# this is for training
# use not just missense
# I do not need to require an mpc score here anymore
df_x_pre = df_pre[ (df_pre.clin_class != 'VUS') & 
                   (df_pre.mpc>0)]
df_s = df_x_pre.groupby('pfam').size().reset_index()
multi_pfam = set( df_s[df_s[0]>1]['pfam'].values )
df_x_pre.loc[:, 'multi_pfam'] = df_x_pre.apply(lambda row: row['pfam'] in multi_pfam, axis=1)
df_x = df_x_pre[df_x_pre.multi_pfam]
df_x.loc[:, 'y'] = df_x.apply(lambda row: 1 if row['clin_class'] in ('PATHOGENIC', 'LIKLEY_PATHOGENIC')
                            else 0, axis=1)
df_x.head()

In [ ]:
clin_file = '../data/interim/clinvar/clinvar.dat'
clinvar_df_pre = pandas.read_csv(clin_file, sep='\t').fillna(0)
def calc_final_sig(row):
    sig_set = set(str(row['clinSig'].split('|')))
    has_benign = '2' in sig_set or '3' in sig_set
    has_path = '4' in sig_set or '5' in sig_set
    if has_path and not has_benign:
        return 1
    if not has_path and has_benign:
        return 0
    return -1

clinvar_df_pre.loc[:, "y"] = clinvar_df_pre.apply(calc_final_sig, axis=1)
clinvar_df = clinvar_df_pre[(clinvar_df_pre.eff=='missense_variant') 
                            & (clinvar_df_pre.y!=-1) 
                            & (clinvar_df_pre.mpc>0)
                            & (clinvar_df_pre.pfam != 'fuck')].drop_duplicates()
clinvar_df.loc[:, 'path_frac_t'] = clinvar_df.apply(lambda row: match(row, domain_info)[0], axis=1)
clinvar_df.loc[:, 'size_t'] = clinvar_df.apply(lambda row: match(row, domain_info)[1], axis=1)
clinvar_df.loc[:, 'path_na_t'] = clinvar_df.apply(lambda row: match(row, domain_info)[2], axis=1)
clinvar_df.loc[:, 'in_none_pfam'] = clinvar_df.apply(lambda row: 1 if 'none' in row['pfam'] else 0, axis=1)
# need a smarter match to domain here
#m = pandas.merge(clinvar_df, ff, on='pfam', how='left')
#m.head()

In [ ]:
print(len(clinvar_df))
print(len(clinvar_df[clinvar_df.y==1]))
print(len(clinvar_df[clinvar_df.y==0]))

In [ ]:
clinvar_df.head()

In [ ]:
df_x[ (df_x.gene=='MECP2') & (df_x['mpc']>0) ].head(20)

In [ ]:
#scores = clinvar_df['mpc'].values
#truth = clinvar_df['y'].values
#fpr_mpc, tpr_mpc, _ = metrics.roc_curve(truth, scores, pos_label=1)
#mpc_auc = metrics.auc(fpr_mpc, tpr_mpc)

In [ ]:
clinvar_df[ (clinvar_df.mpc==0) & (clinvar_df.gene != 'KCTD&') 
           & (clinvar_df.gene != 'FOXG1') 
           & (clinvar_df.gene != 'TSC2') 
          & (clinvar_df.gene != 'MECP2')].groupby('gene').size()

In [ ]:
clinvar_df[ (clinvar_df.mpc==0) & ((clinvar_df.gene == 'KCTD7') 
           | (clinvar_df.gene == 'FOXG1') 
           | (clinvar_df.gene == 'TSC2') 
           | (clinvar_df.gene == 'MECP2'))].groupby('gene').size()

In [ ]:
clinvar_df[ (clinvar_df.gene=='CLN3') & (clinvar_df['mpc']>0) ].head(20)

In [ ]:
# train new tree and apply to clinvar
tree_clf = tree.DecisionTreeClassifier(max_depth=6)
all_preds = []
all_truth = []
cols = ['mpc', 'size_t', 'path_na_t', 'path_frac_t', 'in_none_pfam']
X, y = df_x[cols], df_x['y']
tree_clf.fit(X, y)
dot_data = StringIO()
tree.export_graphviz(tree_clf, feature_names=cols, out_file=dot_data)
graph = pydotplus.graph_from_dot_data( dot_data.getvalue() )
graph.write_pdf('mtr_tree.full.pdf')

X_clin, y_clin = clinvar_df[cols], clinvar_df['y']
preds = tree_clf.predict_proba(X_clin)
fpr_tree, tpr_tree, _ = metrics.roc_curve(y_clin, [x[1] for x in preds], pos_label=1)
tree_auc = metrics.auc(fpr_tree, tpr_tree)

In [ ]:
HTML('<iframe src=./mtr_tree.full.pdf width=1000 height=500></iframe>')

In [ ]:
scores = clinvar_df['mpc'].values
truth = clinvar_df['y'].values
fpr_mpc, tpr_mpc, _ = metrics.roc_curve(truth, scores, pos_label=1)
mpc_auc = metrics.auc(fpr_mpc, tpr_mpc)

In [ ]:
# train new tree and apply to clinvar: just pathogenic frac
tree_clf = tree.DecisionTreeClassifier(max_depth=3)
all_preds = []
all_truth = []
cols = ['size_t', 'path_na_t', 'path_frac_t', 'in_none_pfam']
X, y = df_x[cols], df_x['y']
tree_clf.fit(X, y)
dot_data = StringIO()
tree.export_graphviz(tree_clf, feature_names=cols, out_file=dot_data)
graph = pydotplus.graph_from_dot_data( dot_data.getvalue() )
graph.write_pdf('mtr_tree.full.nompc.pdf')

X_clin, y_clin = clinvar_df[cols], clinvar_df['y']
preds = tree_clf.predict_proba(X_clin)
fpr_tree_nm, tpr_tree_nm, _ = metrics.roc_curve(y_clin, [x[1] for x in preds], pos_label=1)
tree_auc_nm = metrics.auc(fpr_tree_nm, tpr_tree_nm)

In [ ]:
HTML('<iframe src=./mtr_tree.full.nompc.pdf width=1000 height=500></iframe>')

In [ ]:
#X_clin, y_clin = clinvar_df[cols], clinvar_df['y']
#preds = tree_clf.predict_proba(X_clin)
#fpr_tree, tpr_tree, _ = metrics.roc_curve(y_clin, [x[1] for x in preds], pos_label=1)
#tree_auc = metrics.auc(fpr_tree, tpr_tree)
#mpc_auc = metrics.auc(fpr_mpc, tpr_mpc)
print('mpc auc', mpc_auc)
print('tree auc', tree_auc)
print('tree-no-mpc auc', tree_auc_nm)
plt.plot(fpr_tree, tpr_tree, label='tree', color='green')
plt.plot(fpr_tree_nm, tpr_tree_nm, label='tree-no-mpc', color='orange')
plt.plot(fpr_mpc, tpr_mpc, label='mpc', color='black')
plt.legend(loc=4)
plt.title('ClinVar missense variant ROC curve')
plt.savefig('../docs/plots/missense_clinvar_roc_feature_union.png')